In [1]:
!pip install mediapipe

In [2]:
!pip install opencv-python

In [3]:
!pip install pyautogui

In [4]:
!pip install pyperclip

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import socket
import pyperclip
import math
import os
import winsound
import time
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# --- 1. SETTINGS ---
DEST_IP = "10.180.212.222" 
PORT = 5005
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
MODEL_PATH = r"C:\Users\Sneha\Downloads\hand_landmarker.task"

pyautogui.FAILSAFE = False
screen_w, screen_h = pyautogui.size()
prev_x, prev_y = 0, 0
smooth_factor = 0.15
ss_counter = 0
shutdown_counter = 0

# --- 2. DETECTOR SETUP ---
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.HandLandmarkerOptions(base_options=base_options, running_mode=vision.RunningMode.VIDEO, num_hands=1)
detector = vision.HandLandmarker.create_from_options(options)
CONNECTIONS = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),(5,9),(9,10),(10,11),(11,12),(9,13),(13,14),(14,15),(15,16),(13,17),(17,18),(18,19),(19,20),(0,17)]

cap = cv2.VideoCapture(0)
frame_ts = 0

while cap.isOpened():
    success, frame = cap.read()
    if not success: break
    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape
    
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    frame_ts += 1
    result = detector.detect_for_video(mp_image, frame_ts)

    if result.hand_landmarks:
        for landmarks in result.hand_landmarks:
            # SKELETON DRAWING (Green Lines, Red Dots)
            for st_idx, en_idx in CONNECTIONS:
                st, en = landmarks[st_idx], landmarks[en_idx]
                cv2.line(frame, (int(st.x*w), int(st.y*h)), (int(en.x*w), int(en.y*h)), (0, 255, 0), 2)
            for lm in landmarks:
                cv2.circle(frame, (int(lm.x*w), int(lm.y*h)), 5, (0, 0, 255), -1)

            t_t, i_t, m_t, r_t, p_t = landmarks[4], landmarks[8], landmarks[12], landmarks[16], landmarks[20]
            i_base, m_base = landmarks[6], landmarks[10]
            wrist = landmarks[0]

            # Mode Logic
            mouse_m = (i_t.y < i_base.y and m_t.y < m_base.y)
            is_fist = all(math.hypot(landmarks[idx].x - wrist.x, landmarks[idx].y - wrist.y) < 0.2 for idx in [8, 12, 16, 20])
            is_palm = all(math.hypot(landmarks[idx].y - wrist.y) > 0.3 for idx in [8, 12, 16, 20])

            # --- A. MOUSE & TELEPORT ---
            if mouse_m:
                tx, ty = i_t.x * screen_w, i_t.y * screen_h
                curr_x = prev_x + (tx - prev_x) * smooth_factor
                curr_y = prev_y + (ty - prev_y) * smooth_factor
                
                if curr_x > screen_w - 20: # Teleport Trigger
                    data = f"MOVE|{curr_x}|{curr_y}"
                    sock.sendto(data.encode(), (DEST_IP, PORT))
                    cv2.putText(frame, "STATUS: TELEPORTING", (10, 30), 1, 2, (255, 165, 0), 2)
                else:
                    pyautogui.moveTo(curr_x, curr_y, _pause=False)
                    cv2.putText(frame, "STATUS: MOUSE MODE", (10, 30), 1, 2, (0, 255, 0), 2)
                
                prev_x, prev_y = curr_x, curr_y

                # PINCH CLICKS
                if math.hypot(i_t.x - t_t.x, i_t.y - t_t.y) < 0.05:
                    pyautogui.rightClick(); time.sleep(0.3)
                if math.hypot(m_t.x - t_t.x, m_t.y - t_t.y) < 0.05:
                    pyautogui.click(); time.sleep(0.3)

            # --- B. COPY-PASTE LOGIC ---
            elif is_fist: # Mutthi = Copy
                pyautogui.hotkey('ctrl', 'c')
                cv2.putText(frame, "ACTION: COPIED", (w//3, h//2), 1, 2, (0, 255, 0), 3)
            elif is_palm: # Open Palm = Paste
                text = pyperclip.paste()
                data = f"PASTE|{text}"
                sock.sendto(data.encode(), (DEST_IP, PORT))
                cv2.putText(frame, "ACTION: PASTE TO PC-2", (w//4, h//2), 1, 2, (0, 0, 255), 3)

            # --- C. SCROLLING (Index Bend=Up, Middle Bend=Down) ---
            elif not mouse_m:
                if i_t.y > i_base.y + 0.05:
                    pyautogui.scroll(200); cv2.putText(frame, "SCROLL UP", (10, 60), 1, 2, (255, 255, 0), 2)
                elif m_t.y > m_base.y + 0.05:
                    pyautogui.scroll(-200); cv2.putText(frame, "SCROLL DOWN", (10, 60), 1, 2, (255, 255, 0), 2)

            # --- D. SCREENSHOT (Pistol) ---
            if i_t.y < i_base.y and t_t.y < landmarks[2].y and m_t.y > m_base.y:
                ss_counter += 1
                if ss_counter > 20:
                    pyautogui.screenshot(f"ss_{int(time.time())}.png")
                    ss_counter = -20
            else: ss_counter = 0

            # --- E. SHUTDOWN (Pinky + Thumb) ---
            if math.hypot(p_t.x - t_t.x, p_t.y - t_t.y) < 0.04:
                shutdown_counter += 1
                winsound.Beep(1000, 100)
                cv2.putText(frame, f"SHUTDOWN IN: {10-int(shutdown_counter/5)}", (10, 150), 1, 3, (0,0,255), 3)
                if shutdown_counter > 50: os.system("shutdown /s /t 1")
            else: shutdown_counter = 0

    cv2.imshow("Gestro Super Solo (Teleport Edition)", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()